# Deploy a single model on SageMaker Endpoint with Capacity Aware Instance Pools

This notebook demonstrates how to deploy models using **SageMaker Capacity Aware Instance Pools**.

## Feature Overview
- **Instance Pools**: Define up to 5 instance types with priority-based fallback
- **Automatic Capacity Management**: SageMaker automatically selects available capacity
- **LEAST_OUTSTANDING_REQUESTS Routing**: Optimized request distribution across different instance types
- **Autoscaling Support**: Scale across different instance types seamlessly

## Requirements
- **boto3 >= 1.43.1** (released with capacity aware instance pools support)
- **AWS CLI >= 1.45.1** (v1) or **>= 2.34.40** (v2)

## Documentation
- [API Reference](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateEndpointConfig.html)
- [Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-heterogeneous.html)


In [ ]:
# Upgrade boto3 (it will automatically upgrade botocore to the compatible version)
# Note: boto3 1.43.1 was released on 2025-01-XX and may not be available in all pip repositories yet
# If the upgrade fails, try: %pip install --upgrade --index-url https://pypi.org/simple/ boto3==1.43.1
%pip install --upgrade --quiet --no-warn-conflicts 'boto3>=1.35.0'

In [ ]:
# Restart kernel after installing packages
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import json
import time
import boto3
import botocore

region = "us-east-1"

# Check boto3 and botocore versions
try:
    from importlib.metadata import version
    boto3_version = version("boto3")
    botocore_version = version("botocore")
except ImportError:
    boto3_version = boto3.__version__
    botocore_version = botocore.__version__

print(f"Current boto3 version: {boto3_version}")
print(f"Current botocore version: {botocore_version}")

# Verify minimum version requirements
# boto3 1.35.0+ includes support for capacity aware instance pools
min_boto3_version = "1.35.0"

min_boto3_tuple = tuple(map(int, min_boto3_version.split('.')))
current_boto3_tuple = tuple(map(int, boto3_version.split('.')[:3]))

if current_boto3_tuple < min_boto3_tuple:
    print(f"\n❌ ERROR: boto3 version {boto3_version} is too old!")
    print(f"   Capacity aware instance pools require boto3 >= {min_boto3_version}")
    print(f"\n   To upgrade, run this command in a new cell:")
    print(f"   %pip install --upgrade 'boto3>={min_boto3_version}'")
    print(f"\n   Then restart the kernel (Kernel -> Restart Kernel) and re-run from the beginning.")
    raise RuntimeError(f"boto3 version {boto3_version} is too old. Minimum required: {min_boto3_version}")
else:
    print(f"✅ boto3 {boto3_version} is compatible")
    print(f"✅ botocore {botocore_version} is compatible")
    print(f"✅ Ready to use capacity aware instance pools")

# Create standard SageMaker client for production
sm = boto3.client("sagemaker", region_name=region)
print(f"✅ SageMaker client created for region: {region}")

In [ ]:
import re
from IPython.display import clear_output

#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating" or status == "Updating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")




In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container Configuration

Configure the model and container settings for deployment.

In [ ]:
#instance = {"type": "ml.g6e.4xlarge", "num_gpu": 1}
model_id = "Qwen/Qwen3-4B" 
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

# Optional: Specify S3 location if you have pre-downloaded model files
# Replace with your own S3 bucket and prefix
# model_data_source = {
#     'S3DataSource': {
#         'S3Uri': "s3://YOUR-BUCKET-NAME/path/to/model/",
#         'S3DataType': 'S3Prefix',
#         'CompressionType': 'None',
#     }
# }

### LMI (Large Model Inference) Configuration

Using AWS Deep Java Library (DJL) LMI container for optimized inference.
- Supports tensor parallelism for multi-GPU deployment
- Async mode for better throughput
- Optimized for large language models

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi23.0.0-cu129"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": model_id,
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_MAX_MODEL_LEN": "16384",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
}

## Single Model Endpoint (SME) Test

Deploy the model using capacity aware instance pools with automatic capacity fallback.

### Clean up existing resources (if any)

Check and delete any existing resources with the same name to avoid conflicts.

In [ ]:
# Clean up existing resources if they exist
try:
    # Try to delete existing model
    sm.delete_model(ModelName=model_name)
    print(f"✅ Deleted existing model: {model_name}")
except sm.exceptions.ClientError as e:
    if "does not exist" in str(e) or "Could not find" in str(e):
        print(f"ℹ️  No existing model to delete: {model_name}")
    else:
        print(f"⚠️  Error deleting model: {e}")

try:
    # Try to delete existing endpoint
    sm.delete_endpoint(EndpointName=endpoint_name)
    print(f"✅ Deleted existing endpoint: {endpoint_name}")
    # Wait a bit for endpoint deletion to start
    time.sleep(5)
except sm.exceptions.ClientError as e:
    if "does not exist" in str(e) or "Could not find" in str(e):
        print(f"ℹ️  No existing endpoint to delete: {endpoint_name}")
    else:
        print(f"⚠️  Error deleting endpoint: {e}")

try:
    # Try to delete existing endpoint config
    sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"✅ Deleted existing endpoint config: {endpoint_config_name}")
except sm.exceptions.ClientError as e:
    if "does not exist" in str(e) or "Could not find" in str(e):
        print(f"ℹ️  No existing endpoint config to delete: {endpoint_config_name}")
    else:
        print(f"⚠️  Error deleting endpoint config: {e}")

print("\n✅ Ready to create new resources")

In [ ]:
model_response = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
        #"ModelDataSource": model_data_source
    },
)

In [ ]:
endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "VariantInstanceProvisionTimeoutInSeconds": 3600,
            "InstancePools": [
                {"InstanceType": "ml.g6.24xlarge", "Priority": 1},
                {"InstanceType": "ml.g5.48xlarge", "Priority": 2},
                {"InstanceType": "ml.g5.12xlarge", "Priority": 3},
            ]
        },
    ],
)

### Capacity Aware Instance Pool Configuration

**Key Features:**

**RoutingConfig**: Uses **LEAST_OUTSTANDING_REQUESTS (LOR)** routing strategy
- Routes each request to the instance with the fewest in-flight requests per model copy
- Higher-capacity instances process requests faster and drain queues more quickly
- They naturally receive proportionally more traffic without manual weight configuration
- Essential for capacity aware instance pools where instance types differ in throughput capacity

**VariantInstanceProvisionTimeoutInSeconds**: Set to 3600 seconds (1 hour)
- Total window for procuring instances from the pool
- SageMaker retries on Insufficient Capacity errors within this window
- Moves to next priority pool once timeout expires
- Separate from model download and container startup timeouts

**Instance Pools**: Priority-based fallback (up to 3 pools)
- Priority 1 (highest): ml.g6.24xlarge - Latest G6 generation, excellent performance
- Priority 2: ml.g5.48xlarge - Largest G5 instance, highest throughput
- Priority 3 (lowest): ml.g5.12xlarge - Proven G5 reliability, cost-effective

In [ ]:
endpoint_response = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

wait_response = wait_for_endpoint(endpoint_name)

In [ ]:
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
print(endpoint_info)

### Inference Test

Test the deployed endpoint with a sample inference request.

In [ ]:
# Create SageMaker Runtime client for production
sm_runtime = boto3.client("sagemaker-runtime", region_name=region)

payload={
    "messages": [
        {"role": "user", "content": "Name popular places to visit in London?"}
        #{"role": "user", "content": "Solve this problem step by step: What is 15% of 240?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
print(response)


## Autoscaling with Instance Pools


### Register the endpoint for autoscaling

First, we need to register the endpoint variant with Application Auto Scaling.

In [ ]:
# Create Application Auto Scaling client for production
aas = boto3.client("application-autoscaling", region_name=region)

# Register the endpoint variant for autoscaling
resource_id = f"endpoint/{endpoint_name}/variant/{variant_name}"

try:
    register_response = aas.register_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
        MinCapacity=1,
        MaxCapacity=10
    )
    print(f"✅ Registered scalable target for {endpoint_name}")
except Exception as e:
    print(f"❌ Error registering scalable target: {e}")
    raise

### Configure weighted utilization scaling policy

Because the fleet contains instance types with different throughput capacities, we use a weighted metric.
Each term divides a type's observed concurrency by its maximum capacity, producing a value between 0.0 and 1.0.

In [ ]:
# Define max concurrency for each instance type (adjust based on your load testing)
max_concurrency = {
    "ml.g6.24xlarge": 12,
    "ml.g5.48xlarge": 16,
    "ml.g5.12xlarge": 7
}

# Build metrics for each instance type in the pool
metrics = []
metric_ids = []

for idx, pool in enumerate([
    {"InstanceType": "ml.g6.24xlarge", "Priority": 1},
    {"InstanceType": "ml.g5.48xlarge", "Priority": 2},
    {"InstanceType": "ml.g5.12xlarge", "Priority": 3}
]):
    instance_type = pool["InstanceType"]
    metric_id = f"concurrency_{idx}"
    metric_ids.append(metric_id)
    
    metrics.append({
        "Id": metric_id,
        "MetricStat": {
            "Metric": {
                "Namespace": "AWS/SageMaker",
                "MetricName": "ConcurrentRequestsPerModel",
                "Dimensions": [
                    {"Name": "EndpointName", "Value": endpoint_name},
                    {"Name": "VariantName", "Value": variant_name},
                    {"Name": "InstanceType", "Value": instance_type}
                ]
            },
            "Stat": "Average"
        },
        "ReturnData": False
    })

# Build weighted utilization expression
# Each term: concurrency_metric / max_capacity
utilization_terms = []
for idx, pool in enumerate([
    {"InstanceType": "ml.g6.24xlarge", "Priority": 1},
    {"InstanceType": "ml.g5.48xlarge", "Priority": 2},
    {"InstanceType": "ml.g5.12xlarge", "Priority": 3}
]):
    instance_type = pool["InstanceType"]
    max_cap = max_concurrency[instance_type]
    utilization_terms.append(f"concurrency_{idx} / {max_cap}")

num_pools = len(utilization_terms)
expression = f"({" + ".join(utilization_terms)}) / {num_pools}"

metrics.append({
    "Id": "weighted_utilization",
    "Expression": expression,
    "ReturnData": True
})

print(f"Weighted utilization expression: {expression}")

In [ ]:
# Create the target tracking scaling policy
scaling_policy_response = aas.put_scaling_policy(
    PolicyName=f"{endpoint_name}-weighted-utilization-scaling",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 0.7,  # Scale out above 70% weighted fleet utilization
        "ScaleInCooldown": 300,  # Wait 5 minutes before scaling in again
        "ScaleOutCooldown": 60,  # Wait 1 minute before scaling out again
        "CustomizedMetricSpecification": {
            "Metrics": metrics
        }
    }
)

print(f"✅ Autoscaling policy created for {endpoint_name}")

### Describe autoscaling configuration


In [ ]:
# Describe the scalable target
targets = aas.describe_scalable_targets(
    ServiceNamespace="sagemaker",
    ResourceIds=[resource_id],
    ScalableDimension="sagemaker:variant:DesiredInstanceCount"
)

print("Scalable Target:")
print(json.dumps(targets["ScalableTargets"], indent=2, default=str))

# Describe the scaling policies
policies = aas.describe_scaling_policies(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount"
)

print("\nScaling Policies:")
print(json.dumps(policies["ScalingPolicies"], indent=2, default=str))

## Testing Autoscaling

There are several ways to test that autoscaling is working correctly:

### 1. **Verify Configuration** (Quick Check)
Run the describe cells above to confirm:
- Scalable target is registered with MinCapacity=1, MaxCapacity=10
- Scaling policy exists with TargetValue=0.7 (70% utilization)
- Weighted utilization metric is configured for all 3 instance types

### 2. **Monitor CloudWatch Metrics** (Passive Monitoring)
Watch the metrics in CloudWatch console:
- Go to CloudWatch → Metrics → SageMaker
- Look for `ConcurrentRequestsPerModel` for each instance type
- Look for the custom `weighted_utilization` metric
- Monitor `DesiredInstanceCount` and `CurrentInstanceCount`

### 3. **Load Testing** (Active Testing)
Generate load to trigger scale-out:
- Send concurrent requests to push utilization above 70%
- Watch instance count increase (takes 1-3 minutes)
- Stop load and watch scale-in after cooldown (5 minutes)

### 4. **Manual Scaling Test** (Verification)
Manually adjust desired capacity to verify autoscaling responds:
- Set desired count to 3
- Verify endpoint scales to 3 instances
- Autoscaling will adjust based on actual load

Below are code examples for each testing method.

###  Monitor CloudWatch Metrics

Query CloudWatch metrics programmatically to see current utilization.

In [ ]:
# Manually set desired capacity to test scaling
TEST_CAPACITY = 3

print(f"🔧 Manually setting desired capacity to {TEST_CAPACITY} instances...\n")

try:
    # Update the variant to scale to TEST_CAPACITY instances
    update_response = sm.update_endpoint_weights_and_capacities(
        EndpointName=endpoint_name,
        DesiredWeightsAndCapacities=[
            {
                'VariantName': variant_name,
                'DesiredInstanceCount': TEST_CAPACITY
            }
        ]
    )
    
    print(f"✅ Update initiated. Endpoint will scale to {TEST_CAPACITY} instances.")
    print(f"\n⏳ Scaling takes 3-8 minutes depending on instance availability.")
    print(f"\n💡 Monitor progress:")
    print(f"   - Run the CloudWatch metrics cell above")
    print(f"   - Check AWS Console → SageMaker → Endpoints → {endpoint_name}")
    print(f"\n⚠️  Remember: Autoscaling will adjust capacity based on actual load.")
    print(f"   If load is low, it will scale back down after the cooldown period (5 min).")
    
except Exception as e:
    print(f"❌ Error updating capacity: {e}")

In [ ]:
from datetime import datetime, timedelta, timezone

cloudwatch = boto3.client('cloudwatch', region_name=region)

# Get metrics for the last 10 minutes
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=10)

print("📊 CloudWatch Metrics (Last 10 minutes)\n")

# Check concurrent requests per instance type
for instance_type in ["ml.g6.24xlarge", "ml.g5.48xlarge", "ml.g5.12xlarge"]:
    response = cloudwatch.get_metric_statistics(
        Namespace='AWS/SageMaker',
        MetricName='ConcurrentRequestsPerModel',
        Dimensions=[
            {'Name': 'EndpointName', 'Value': endpoint_name},
            {'Name': 'VariantName', 'Value': variant_name},
            {'Name': 'InstanceType', 'Value': instance_type}
        ],
        StartTime=start_time,
        EndTime=end_time,
        Period=60,  # 1 minute intervals
        Statistics=['Average', 'Maximum']
    )
    
    if response['Datapoints']:
        latest = sorted(response['Datapoints'], key=lambda x: x['Timestamp'])[-1]
        print(f"  {instance_type}:")
        print(f"    Average: {latest.get('Average', 0):.2f} concurrent requests")
        print(f"    Maximum: {latest.get('Maximum', 0):.2f} concurrent requests")
    else:
        print(f"  {instance_type}: No data yet (endpoint may be new)")

# Check current instance count
endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
for variant in endpoint_info['ProductionVariants']:
    if variant['VariantName'] == variant_name:
        print(f"\n📈 Current Instance Count: {variant['CurrentInstanceCount']}")
        print(f"   Desired Instance Count: {variant['DesiredInstanceCount']}")
        if 'InstancePools' in variant:
            print(f"\n   Instance Distribution:")
            for pool in variant['InstancePools']:
                print(f"     - {pool['InstanceType']}: {pool.get('CurrentInstanceCount', 0)} instances")

In [ ]:
# Watch endpoint scaling progress
print("👀 Monitoring endpoint scaling (press Ctrl+C to stop)...\n")

try:
    for i in range(20):  # Monitor for up to 20 iterations (10 minutes)
        endpoint_info = sm.describe_endpoint(EndpointName=endpoint_name)
        
        for variant in endpoint_info['ProductionVariants']:
            if variant['VariantName'] == variant_name:
                current = variant['CurrentInstanceCount']
                desired = variant['DesiredInstanceCount']
                status = endpoint_info['EndpointStatus']
                
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Status: {status} | "
                      f"Current: {current} | Desired: {desired}")
                
                if 'InstancePools' in variant:
                    for pool in variant['InstancePools']:
                        pool_count = pool.get('CurrentInstanceCount', 0)
                        if pool_count > 0:
                            print(f"  └─ {pool['InstanceType']}: {pool_count} instances")
                
                if current == desired and status == 'InService':
                    print(f"\n✅ Scaling complete! Endpoint has {current} instances.")
                    break
        else:
            time.sleep(30)  # Wait 30 seconds before next check
            continue
        break
        
except KeyboardInterrupt:
    print("\n⏹️  Monitoring stopped by user.")

### Testing Summary

**Expected Behavior:**

1. **Scale-Out Trigger:**
   - When weighted utilization > 70% for ~1 minute
   - Autoscaling adds instances (respects MaxCapacity=10)
   - New instances come from priority pools (P1 → P2 → P3)
   - Takes 1-3 minutes to react + 3-8 minutes to provision

2. **Scale-In Trigger:**
   - When weighted utilization < 70% for ~5 minutes (cooldown)
   - Autoscaling removes instances (respects MinCapacity=1)
   - Takes 5+ minutes due to scale-in cooldown

3. **Instance Pool Behavior:**
   - SageMaker tries Priority 1 (ml.g6.24xlarge) first
   - Falls back to Priority 2 (ml.g5.48xlarge) if P1 unavailable
   - Falls back to Priority 3 (ml.g5.12xlarge) if P2 unavailable
   - Can mix instance types across the fleet

4. **Routing:**
   - LEAST_OUTSTANDING_REQUESTS sends traffic to least busy instance
   - Higher-capacity instances naturally get more traffic
   - No manual weight configuration needed

**Troubleshooting:**

- **No scaling happening?**
  - Check CloudWatch metrics are being published
  - Verify utilization actually exceeds 70%
  - Wait at least 3-5 minutes for autoscaling to react
  
- **Scaling too slow?**
  - Reduce ScaleOutCooldown (currently 60s)
  - Lower TargetValue threshold (currently 0.7)
  
- **Scaling too aggressive?**
  - Increase TargetValue threshold
  - Increase ScaleOutCooldown
  
- **InsufficientCapacity errors?**
  - Check if all 3 instance types are available in your region
  - Increase VariantInstanceProvisionTimeoutInSeconds (currently 3600s)
  - Add more instance types to the pool (up to 5 total)

### Autoscaling Cleanup


In [ ]:
# Delete the scaling policy
delete_policy_response = aas.delete_scaling_policy(
    PolicyName=f"{endpoint_name}-weighted-utilization-scaling",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount"
)

# Deregister the scalable target
deregister_response = aas.deregister_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount"
)

print("✅ Autoscaling configuration removed")

### SME Cleanup

Clean up resources to avoid ongoing charges.
**Important:** Run this cell when you're done testing to delete the endpoint, endpoint config, and model.

In [ ]:
delete_endpoint_response = sm.delete_endpoint(EndpointName=endpoint_name)
delete_config_response = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
#delete_scale_config_response = sm.delete_endpoint_config(EndpointConfigName=scale_out_config_name)
delete_model_response = sm.delete_model(ModelName=model_name)